In [ ]:
pip install requests pandas praw python-dotenv

In [ ]:
pip install beautifulsoup4 lxml

In [ ]:
import requests
import pandas as pd

appid = "238960"  # Path of Exile
url = f"https://store.steampowered.com/appreviews/{appid}"

params = {
    "json": 1,
    "num_per_page": 20,
    "language": "all",
    "filter": "recent"
}

resp = requests.get(url, params=params)
data = resp.json()

print(data.keys())
print("评论条数:", len(data["reviews"]))

for r in data["reviews"][:3]:
    print("-----")
    print("推荐:", r["voted_up"])
    print("游玩时长(分钟):", r["author"]["playtime_forever"])
    print("评论前200字:", r["review"][:200])

In [ ]:
!python steam_test.py

In [ ]:
import requests
import pandas as pd
import time

def fetch_steam_reviews(appid: str, max_reviews: int = 1500, language: str = "english") -> pd.DataFrame:
    """
    Fetch Steam reviews for a given appid.
    
    Parameters:
        appid (str): Steam app ID
        max_reviews (int): Maximum number of reviews to fetch
        language (str): Review language, e.g. "english"
    
    Returns:
        pd.DataFrame: Reviews dataframe
    """
    url = f"https://store.steampowered.com/appreviews/{appid}"
    cursor = "*"
    rows = []

    while len(rows) < max_reviews:
        params = {
            "json": 1,
            "cursor": cursor,
            "num_per_page": 100,
            "language": language,
            "filter": "updated",
            "purchase_type": "all"
        }

        try:
            response = requests.get(url, params=params, timeout=30)
            response.raise_for_status()
            data = response.json()
        except Exception as e:
            print(f"Request failed: {e}")
            break

        reviews = data.get("reviews", [])
        if not reviews:
            print("No more reviews returned.")
            break

        for r in reviews:
            rows.append({
                "game": "Path of Exile",
                "platform": "Steam",
                "review_id": r.get("recommendationid"),
                "review_text": r.get("review"),
                "voted_up": r.get("voted_up"),
                "votes_up": r.get("votes_up"),
                "votes_funny": r.get("votes_funny"),
                "weighted_vote_score": r.get("weighted_vote_score"),
                "comment_count": r.get("comment_count"),
                "steam_purchase": r.get("steam_purchase"),
                "received_for_free": r.get("received_for_free"),
                "written_during_early_access": r.get("written_during_early_access"),
                "timestamp_created": r.get("timestamp_created"),
                "timestamp_updated": r.get("timestamp_updated"),
                "playtime_forever": r.get("author", {}).get("playtime_forever"),
                "playtime_last_two_weeks": r.get("author", {}).get("playtime_last_two_weeks"),
                "playtime_at_review": r.get("author", {}).get("playtime_at_review"),
                "last_played": r.get("author", {}).get("last_played"),
            })

            if len(rows) >= max_reviews:
                break

        cursor = data.get("cursor", cursor)
        print(f"Fetched {len(rows)} reviews so far...")

        time.sleep(1)

    df = pd.DataFrame(rows)

    if not df.empty:
        df["timestamp_created"] = pd.to_datetime(df["timestamp_created"], unit="s", errors="coerce")
        df["timestamp_updated"] = pd.to_datetime(df["timestamp_updated"], unit="s", errors="coerce")
        df["last_played"] = pd.to_datetime(df["last_played"], unit="s", errors="coerce")

        df["playtime_hours"] = df["playtime_forever"] / 60
        df["playtime_at_review_hours"] = df["playtime_at_review"] / 60
        df["review_length"] = df["review_text"].astype(str).str.len()

        df = df.drop_duplicates(subset=["review_id"])
        df = df.dropna(subset=["review_text"])
        df["review_text"] = df["review_text"].astype(str).str.strip()
        df = df[df["review_text"] != ""]

    return df


# Path of Exile appid
POE_APPID = "238960"

df_poe = fetch_steam_reviews(appid=POE_APPID, max_reviews=1500, language="english")

print("\nFinished!")
print(df_poe.shape)
print(df_poe.head())

df_poe.to_csv("poe_steam_reviews_english_1500.csv", index=False, encoding="utf-8-sig")
print("Saved to poe_steam_reviews_english_1500.csv")

In [ ]:
print(df_poe[["review_id", "review_text", "voted_up", "playtime_hours"]].head())
print(df_poe["voted_up"].value_counts(normalize=True))
print(df_poe["review_length"].describe())

In [ ]:
df_poe_clean = df_poe.copy()
df_poe_clean.to_csv("poe_steam_reviews_english_1493_clean.csv", index=False, encoding="utf-8-sig")
print("clean version saved")

In [ ]:
import pandas as pd

df = pd.read_csv("poe_steam_reviews_english_1500.csv")

# 如果你还没转时间，补一下
df["timestamp_created"] = pd.to_datetime(df["timestamp_created"], errors="coerce")

# 再确认几个核心指标
print(df["voted_up"].value_counts())
print(df["voted_up"].value_counts(normalize=True))

print("\nPlaytime by recommendation:")
print(df.groupby("voted_up")["playtime_hours"].describe())

print("\nReview length by recommendation:")
print(df.groupby("voted_up")["review_length"].describe())

In [ ]:
import matplotlib.pyplot as plt

q95 = df["playtime_hours"].quantile(0.95)
df_trim = df[df["playtime_hours"] <= q95]

df_trim.boxplot(column="playtime_hours", by="voted_up")
plt.title("Path of Exile: Playtime Hours by Recommendation")
plt.suptitle("")
plt.xlabel("Recommended")
plt.ylabel("Playtime Hours")
plt.show()

In [ ]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", " ", text)          # 去链接
    text = re.sub(r"[^a-zA-Z\s]", " ", text)      # 只保留英文和空格
    text = re.sub(r"\s+", " ", text).strip()      # 多空格合并
    return text

df["clean_text"] = df["review_text"].apply(clean_text)
df[["review_text", "clean_text"]].head()

In [ ]:
from collections import Counter
import re

target_phrases = {
    "skill tree",
    "passive skill",
    "learning curve",
    "endgame content",
    "build diversity",
    "league start",
    "league mechanic",
    "league launch",
    "server issues",
    "fix servers",
    "servers unplayable",
    "loot filter",
    "stash tabs",
    "come back",
    "grind grind",
    "hack slash",
    "builds viable",
    "skill gems",
    "new league",
    "trade system",
    "crafting system",
    "boss fights",
    "game crashes",
    "performance issues",
    "map sustain"
}

def count_target_phrases(text_series, phrase_set):
    counter = Counter()
    for text in text_series.dropna():
        text = str(text).lower()
        for phrase in phrase_set:
            # 精确匹配短语
            if phrase in text:
                counter[phrase] += 1
    return counter

positive_phrase_counts = count_target_phrases(
    df[df["voted_up"] == True]["clean_text"], target_phrases
)

negative_phrase_counts = count_target_phrases(
    df[df["voted_up"] == False]["clean_text"], target_phrases
)

print("Positive phrase counts:")
print(positive_phrase_counts.most_common())

print("\nNegative phrase counts:")
print(negative_phrase_counts.most_common())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pos_phrase_df = pd.DataFrame(positive_phrase_counts.most_common(), columns=["phrase", "count"])
neg_phrase_df = pd.DataFrame(negative_phrase_counts.most_common(), columns=["phrase", "count"])

plt.figure(figsize=(10,6))
plt.barh(pos_phrase_df["phrase"][::-1], pos_phrase_df["count"][::-1])
plt.title("Path of Exile: Retention-Related Phrases in Positive Reviews")
plt.xlabel("Count")
plt.ylabel("Phrase")
plt.show()

plt.figure(figsize=(10,6))
plt.barh(neg_phrase_df["phrase"][::-1], neg_phrase_df["count"][::-1])
plt.title("Path of Exile: Pain Points in Negative Reviews")
plt.xlabel("Count")
plt.ylabel("Phrase")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 选你最想比较的核心短语
focus_phrases = [
    "skill tree",
    "build diversity",
    "endgame content",
    "come back",
    "new league",
    "league start",
    "server issues",
    "league mechanic"
]

# 把 Counter 转成 dict
pos_dict = dict(positive_phrase_counts)
neg_dict = dict(negative_phrase_counts)

# 组表
compare_df = pd.DataFrame({
    "phrase": focus_phrases,
    "positive": [pos_dict.get(p, 0) for p in focus_phrases],
    "negative": [neg_dict.get(p, 0) for p in focus_phrases]
})

print(compare_df)

# 画 grouped bar chart
x = np.arange(len(compare_df))
width = 0.38

plt.figure(figsize=(12, 6))
plt.bar(x - width/2, compare_df["positive"], width=width, label="Positive Reviews")
plt.bar(x + width/2, compare_df["negative"], width=width, label="Negative Reviews")

plt.xticks(x, compare_df["phrase"], rotation=30, ha="right")
plt.ylabel("Count")
plt.xlabel("Phrase")
plt.title("Path of Exile: Positive vs Negative Review Themes")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
compare_df = compare_df.sort_values("positive", ascending=True)

y = np.arange(len(compare_df))
height = 0.38

plt.figure(figsize=(10, 6))
plt.barh(y - height/2, compare_df["positive"], height=height, label="Positive Reviews")
plt.barh(y + height/2, compare_df["negative"], height=height, label="Negative Reviews")

plt.yticks(y, compare_df["phrase"])
plt.xlabel("Count")
plt.ylabel("Phrase")
plt.title("Path of Exile: Positive vs Negative Review Themes")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
df_poe["playtime_hours"] = df_poe["playtime_forever"] / 60

In [ ]:
print(df_poe.groupby("voted_up")["playtime_hours"].describe())
print("\nMedian playtime by review outcome:")
print(df_poe.groupby("voted_up")["playtime_hours"].median())

In [ ]:
import pandas as pd
import numpy as np

# 你可以按 PoE 的特点用更适合重度玩家的分组
bins = [0, 10, 50, 100, 300, 1000, np.inf]
labels = ["0–10h", "10–50h", "50–100h", "100–300h", "300–1000h", "1000h+"]

df_poe["playtime_bin"] = pd.cut(
    df_poe["playtime_hours"],
    bins=bins,
    labels=labels,
    right=False
)

# 每组评论数 + 好评率
poe_bin_summary = (
    df_poe.groupby("playtime_bin")
    .agg(
        review_count=("voted_up", "count"),
        positive_rate=("voted_up", "mean"),
        median_hours=("playtime_hours", "median")
    )
    .reset_index()
)

print(poe_bin_summary)

In [ ]:
import matplotlib.pyplot as plt

plot_df = poe_bin_summary.dropna(subset=["positive_rate"]).copy()

plt.figure(figsize=(9, 5.5))
plt.bar(plot_df["playtime_bin"].astype(str), plot_df["positive_rate"])
plt.ylim(0, 1)
plt.xlabel("Playtime Group")
plt.ylabel("Positive Review Rate")
plt.title("Path of Exile: Positive Review Rate by Playtime Group")
plt.tight_layout()
plt.show()

In [ ]:
thresholds = [100, 300, 1000]

for t in thresholds:
    share_all = (df_poe["playtime_hours"] >= t).mean()
    share_by_sentiment = df_poe.groupby("voted_up").apply(
        lambda x: (x["playtime_hours"] >= t).mean()
    )
    
    print(f"\n=== {t}+ hours ===")
    print(f"All reviews: {share_all:.2%}")
    print("By review outcome:")
    print(share_by_sentiment)

In [ ]:
poe_sentiment_bin = (
    df_poe.groupby(["playtime_bin", "voted_up"])
    .size()
    .unstack(fill_value=0)
)

# 转成比例
poe_sentiment_bin_pct = poe_sentiment_bin.div(poe_sentiment_bin.sum(axis=1), axis=0)

poe_sentiment_bin_pct.plot(
    kind="bar",
    stacked=True,
    figsize=(9, 5.5)
)

plt.xlabel("Playtime Group")
plt.ylabel("Share of Reviews")
plt.title("Path of Exile: Review Outcome Composition by Playtime Group")
plt.legend(title="Recommended", labels=["Negative", "Positive"])
plt.tight_layout()
plt.show()

In [ ]:
short_negative_poe = df_poe[
    (df_poe["playtime_bin"] == "0–10h") &
    (df_poe["voted_up"] == False)
][["review_text", "playtime_hours"]]

print(short_negative_poe.head(30))

In [ ]:
def classify_short_negative(text):
    t = str(text).lower()

    if any(k in t for k in ["crash", "server", "network", "unplayable", "mac", "performance"]):
        return "technical/performance"
    elif any(k in t for k in ["beginner", "hard to understand", "confusing", "slow to get started"]):
        return "onboarding/complexity"
    elif any(k in t for k in ["locked", "ban", "account"]):
        return "account/platform issue"
    elif any(k in t for k in ["hype", "overhyped", "dont see what the hype is about"]):
        return "expectation mismatch"
    elif len(t.strip()) <= 10:
        return "low-information"
    else:
        return "other"

In [ ]:
short_negative_poe["early_negative_reason"] = short_negative_poe["review_text"].apply(classify_short_negative)
print(short_negative_poe["early_negative_reason"].value_counts())

In [ ]:
long_negative_poe = df_poe[
    (df_poe["playtime_hours"] >= 1000) &
    (df_poe["voted_up"] == False)
][["review_text", "playtime_hours"]].sort_values("playtime_hours", ascending=False)

print(long_negative_poe.head(30))

In [ ]:
mid_long_negative_poe = df_poe[
    (df_poe["playtime_hours"] >= 300) &
    (df_poe["playtime_hours"] < 1000) &
    (df_poe["voted_up"] == False)
][["review_text", "playtime_hours"]].sort_values("playtime_hours", ascending=False)

print(mid_long_negative_poe.head(30))